# 61 — SID Training Data (W2)

Combines raw conversations + metadata-as-query + (optional) doc2query into a single stratified train/val parquet for W3 generator fine-tune.

**Inputs**:
- W1 artifact `experiments/cache/sid/track_to_sid.parquet` (must exist on Drive — run notebook 60 first)
- HF `talkpl-ai/TalkPlayData-Challenge-Dataset[train]`
- HF `talkpl-ai/TalkPlayData-Challenge-Track-Metadata[all_tracks]`
- (Optional) `experiments/cache/doc2query/.../queries.parquet` (run notebook 54 first if you want this source)

**Outputs** (cached on Drive at `recsys2026_sid_training_cache/`, NOT committed to git — `experiments/cache/` is gitignored):
- `experiments/cache/sid_training/train.parquet`
- `experiments/cache/sid_training/val.parquet`
- `experiments/cache/sid_training/summary.json`

**Wallclock**: ~5–10 min on L4 (no GPU needed; mostly HF dataset download + dataframe building).

In [ ]:
# 1) Clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 2) HF auth + Drive mount + symlinks for SID + doc2query + sid_training caches.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

for repo_path, drive_subdir in [
    ('experiments/cache/sid', 'recsys2026_sid_cache'),
    ('experiments/cache/doc2query', 'recsys2026_doc2query_cache'),
    ('experiments/cache/sid_training', 'recsys2026_sid_training_cache'),
]:
    drive_path = f'/content/drive/MyDrive/{drive_subdir}'
    full_repo = f'/content/recsys2026/{repo_path}'
    os.makedirs(drive_path, exist_ok=True)
    os.makedirs(os.path.dirname(full_repo), exist_ok=True)
    if os.path.lexists(full_repo):
        !rm -rf {full_repo}
    !ln -s {drive_path} {full_repo}
    print(f'symlinked {full_repo} -> {drive_path}')

In [ ]:
# 3) Install deps (no GPU work — minimal stack).
!pip install -q --upgrade datasets 'pandas<3.0' tqdm pyarrow

In [ ]:
# 4) Verify W1 SID lookup is on Drive.
import os
sid_path = 'experiments/cache/sid/track_to_sid.parquet'
if not os.path.exists(sid_path):
    raise FileNotFoundError(f'{sid_path} missing — run notebook 60 (W1) first')
import pandas as pd
sid_df = pd.read_parquet(sid_path)
print(f'W1 SID lookup: {len(sid_df)} tracks, columns={sid_df.columns.tolist()}')

In [ ]:
# 5) (Optional smoke) Build training data on first 100 train sessions only — fast sanity check.
!python scripts/build_sid_training_data.py --max-sessions 100 --no-doc2query

In [ ]:
# 5b) Inspect the smoke output.
import json, pandas as pd
summary = json.load(open('experiments/cache/sid_training/summary.json'))
print(json.dumps(summary, indent=2))
train = pd.read_parquet('experiments/cache/sid_training/train.parquet')
val = pd.read_parquet('experiments/cache/sid_training/val.parquet')
print(f'\ntrain shape: {train.shape}')
print(f'val shape: {val.shape}')
print('\ntrain head:')
print(train.head(3))

In [ ]:
# 6) Full run — all train sessions + metadata. doc2query DISABLED for v1.
#
# Why no doc2query: the existing doc2query parquet was audited 2026-05-16 and found
# to have 3 critical quality issues (refusal rows survive parsing, prompt invites
# metadata copy-paste → entity-lookup queries instead of conversational, only 2K of
# 47K tracks covered due to early stop). Training W3 on those queries would teach
# the SID generator the wrong query→SID mapping.
#
# 168K raw+metadata pairs is plenty for W3 v1. If W3 gate fails, regenerate doc2query
# (notebook 54 with prompt + parser fixes) and remove the flag below for W3 v2.
#
# Wallclock: ~5-10 min, mostly HF dataset download (cached after first run).
!python scripts/build_sid_training_data.py --no-doc2query

In [ ]:
# 7) Final inspection.
import json, pandas as pd
summary = json.load(open('experiments/cache/sid_training/summary.json'))
print(json.dumps(summary, indent=2))
print('\nper-source counts in train:')
train = pd.read_parquet('experiments/cache/sid_training/train.parquet')
print(train['source'].value_counts())

## 8) Outputs

Outputs are on Drive at `recsys2026_sid_training_cache/` — no git commit needed. `experiments/cache/` is gitignored, and the symlink in cell 2 mirrors the Drive folder into the repo path so the orchestration script and W3's dataloader can read it transparently. Re-running this notebook on a fresh Colab runtime will re-mount Drive and re-establish the symlink before any reads.